<a href="https://colab.research.google.com/github/Ashwini9713/fundamentals-of-ML/blob/main/exp3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Decision Tree ID3 Algorithm Demonstration
This notebook demonstrates the working of the ID3 (Iterative Dichotomiser 3) decision tree algorithm. We will:
1. Load the `play_tennis (1).xlsx` dataset.
2. Build the ID3 algorithm using Shannon Entropy and Information Gain.
3. Fit the tree on our dataset.
4. Classify a new sample.

In [1]:
import pandas as pd
import numpy as np

# Load the dataset
file_path = '/content/play_tennis (1).xlsx'
df = pd.read_excel(file_path)

# Display the dataset
display(df)

,Outlook,Temperature,Humidity,Wind,Target
0,sunny,hot,high,weak,no
1,sunny,hot,high,strong,no
2,overcast,hot,high,weak,yes
3,rain,mild,high,weak,yes
4,rain,cool,normal,weak,yes
5,rain,cool,normal,strong,no
6,overcast,cool,normal,strong,yes
7,sunny,mild,high,weak,no
8,sunny,cool,normal,weak,yes
9,rain,mild,normal,weak,yes


In [2]:
def entropy(target_col):
    elements, counts = np.unique(target_col, return_counts=True)
    entropy_val = np.sum([(-counts[i]/np.sum(counts)) * np.log2(counts[i]/np.sum(counts)) for i in range(len(elements))])
    return entropy_val

def info_gain(data, split_attribute_name, target_name="PlayTennis"):
    total_entropy = entropy(data[target_name])
    vals, counts= np.unique(data[split_attribute_name], return_counts=True)
    Weighted_Entropy = np.sum([(counts[i]/np.sum(counts)) * entropy(data[data[split_attribute_name]==vals[i]][target_name]) for i in range(len(vals))])
    Information_Gain = total_entropy - Weighted_Entropy
    return Information_Gain

def ID3(data, original_data, features, target_attribute_name="PlayTennis", parent_node_class = None):
    # If all target values are the same, return this value
    if len(np.unique(data[target_attribute_name])) <= 1:
        return np.unique(data[target_attribute_name])[0]

    # If dataset is empty, return the mode target value in the original dataset
    elif len(data) == 0:
        return np.unique(original_data[target_attribute_name])[np.argmax(np.unique(original_data[target_attribute_name], return_counts=True)[1])]

    # If features space is empty, return the mode target value of the parent node
    elif len(features) == 0:
        return parent_node_class

    # Otherwise, build the tree
    else:
        # Determine the parent node class
        parent_node_class = np.unique(data[target_attribute_name])[np.argmax(np.unique(data[target_attribute_name], return_counts=True)[1])]

        # Select the feature which best splits the data
        item_values = [info_gain(data, feature, target_attribute_name) for feature in features]
        best_feature_index = np.argmax(item_values)
        best_feature = features[best_feature_index]

        # Create the tree structure
        tree = {best_feature: {}}

        # Remove the feature with the best info gain
        features = [i for i in features if i != best_feature]

        # Grow a branch for each value of the best feature
        for value in np.unique(data[best_feature]):
            sub_data = data[data[best_feature] == value]
            subtree = ID3(sub_data, original_data, features, target_attribute_name, parent_node_class)
            tree[best_feature][value] = subtree

        return tree

In [3]:
# Prepare features and target names (adjust target column name if spelling differs in the file)
target_col = 'PlayTennis' if 'PlayTennis' in df.columns else df.columns[-1]
feature_cols = [col for col in df.columns if col not in [target_col, 'Day', 'day', 'id', 'ID']]

# Build the tree
tree = ID3(df, df, feature_cols, target_col)
import pprint
pprint.pprint(tree)

{'Outlook': {'overcast': 'yes',
             'rain': {'Wind': {'strong': 'no', 'weak': 'yes'}},
             'sunny': {'Humidity': {'high': 'no', 'normal': 'yes'}}}}


In [5]:
def predict(query, tree, default = 'Yes'):
    # Normalize the query keys and values to lowercase to match the tree keys
    normalized_query = {k.lower(): str(v).lower() for k, v in query.items()}

    # Traverse the tree
    current_node = tree
    while isinstance(current_node, dict):
        # Get the splitting feature
        feature = list(current_node.keys())[0]
        feature_lower = feature.lower()

        if feature_lower in normalized_query:
            val = normalized_query[feature_lower]
            if val in current_node[feature]:
                current_node = current_node[feature][val]
            else:
                return default
        else:
            return default

    return current_node

# Predict using the original sample with case normalization
new_sample = {'Outlook': 'Rain', 'Temperature': 'Cool', 'Humidity': 'Normal', 'Wind': 'Strong'}
prediction = predict(new_sample, tree)
print(f"Prediction for sample {new_sample}: {prediction}")

Prediction for sample {'Outlook': 'Rain', 'Temperature': 'Cool', 'Humidity': 'Normal', 'Wind': 'Strong'}: no
